## sources:

SEER*Rx Interactive Antineoplastic Drugs Database:
drugs:
https://seer.cancer.gov/seertools/seerrx/#

extensive list of drugs and regiments.

need to manually add treatment keywords. 

in this notebook, we will use regex on drug and treament names to find samples for which we also know that they were treated by something specific

In [ ]:
# import
import polars as pl
import pandas as pd
import re
from pathlib import Path
from tqdm import tqdm
from fastcore.all import *
# Paths
data_dir = Path('../data')

In [ ]:
list(data_dir.iterdir())

[Path('../data/experiment_data.csv'),
 Path('../data/clin_obs.csv'),
 Path('../data/response_column_shortlist_lazy.parquet'),
 Path('../data/novel_samples.csv'),
 Path('../data/hihg_potential_clinical_samples.csv'),
 Path('../data/approved_experiments.csv'),
 Path('../data/experiment_data_with_scores_labelled.csv'),
 Path('../data/potential_clin_data.csv'),
 Path('../data/metadata_HOMO_SAPIENS.parquet'),
 Path('../data/candidate_experiments.csv'),
 Path('../data/manualu evaluation.md'),
 Path('../data/response_column_shortlist.parquet'),
 Path('../data/clean_score_df.csv'),
 Path('../data/experiment_data_with_scores.csv'),
 Path('../data/drugs.csv'),
 Path('../data/regimens.csv')]

In [ ]:
print(Path('../data/manualu evaluation.md').read_text())

after  evaluating 40 outputs:

# conclusions:
1. PR as a collumn is a red herring. can be progresterone receptor status.
2. cancer score col > 1, or refine bio disease, or cancer text > 1
3. search sample metadata for drug or treatment names. neoadjuvant. chemo. etc. need to have a large source. 


# source: 
## evaluted:
=== TOP 40 EXPERIMENTS WITH RESPONSE DATA ===

1. GSE21653 (262 samples)
   Title: A gene expression signature identifies two prognostic subgroups of basal breast cancer
   overall design: The IPC series contained frozen tumor samples obtained from 266 early breast cancer patients who underwent initial surgery in our institution between 1992 and 2004. They included 227 cases previously reported {Finetti, 2008 #1758} and 39 additional cases, all similarly profiled using Affymetrix U133 Plus 2.0 human oligonucleotide microarrays as previously described {Finetti, 2008 #1758}. The study was approved by the IPC review board, and informed consent was available for each case

In [ ]:
manual_ids = L(['GSE16716', 'GSE37180', 'GSE46819', 'GSE25055', 'GSE40020', 'GSE25066', 
 'GSE15622', 'GSE18864', 'GSE31852', 'GSE106977', 'GSE29561', 'GSE35640', 
 'GSE28826', 'GSE84334', 'GSE93984', 'GSE28583', 'GSE95770', 'GSE28694'])

# load data

In [ ]:
samples = pl.read_csv(data_dir / 'hihg_potential_clinical_samples.csv',infer_schema_length=None)


In [ ]:
samples.columns[:2],samples.columns[-12:]

(['refinebio_accession_code', 'experiment_accession'],
 ['series_title',
  'series_summary',
  'series_overall_design',
  'series_relation',
  'series_n_samples',
  'series_platform_id',
  'series_is_superseries',
  'primary_score',
  'cell_line_score',
  'net_score',
  'score_cancer_cols',
  'score_cancer_text'])

In [ ]:
score_subset = samples.select(samples.columns[:2] + samples.columns[-12:])

load these two data sets and display them

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I can help you load those datasets! Based on your data directory, I can see you have `drugs.csv` and `regimens.csv`. Here's how to load them:

```python
drugs = pl.read_csv(data_dir / 'drugs.csv')
regimens = pl.read_csv(data_dir / 'regimens.csv')
```

Then to display them, you could use:

```python
drugs
```

and 

```python
regimens
```

Would you like to run this code, or would you prefer to explore them in a different way (like checking their shape, columns, or a sample of rows first)?

In [ ]:
drugs = pl.read_csv(data_dir / 'drugs.csv')
regimens = pl.read_csv(data_dir / 'regimens.csv')

In [ ]:
drugs

Name,EVS ID,Histology,Do not code,Remarks,Alternate Name,Primary Site,Abbreviation,NSC number,Category,Sub-category
str,str,str,str,str,str,str,str,str,str,str
"""131I-MIBG""",null,"""pheochromocytoma, paragangliom…","""SEE_REMARKS""","""Do NOT code if given as a radi…","""I131 metaiodobenzylguanidine""",null,null,null,"""Radiation""","""Radioisotope"""
"""5-Aza-2'Deoxy-cytidine""",null,null,"""NO""",null,"""Decytabine""","""CML""","""DAC""",null,"""Chemotherapy""","""Cytostatic agent--DNA methyl t…"
"""5-FP""",null,null,"""NO""","""Oral 5-FU prodrug that is conv…","""5-fluoro-Pyrimidinone""","""Breast;gastric;head & neck;liv…",null,null,"""Chemotherapy""","""Antimetabolite"""
"""ABT 888""",null,null,"""NO""","""Dec. 2013: received FDA approv…","""Veliparib""","""Ovary""",null,null,"""Chemotherapy""","""PARP Inhibitor"""
"""ADXS-HPV""",null,null,"""NO""","""The FDA granted Orphan Drug de…",null,"""Cervix""",null,null,"""Biologic therapy (BRM, immunot…",null
…,…,…,…,…,…,…,…,…,…,…
"""rgp120 CHO""",null,null,"""YES""",null,"""Biocine""",null,null,null,"""AIDS drug""",null
"""rgp120/HIV-1 (IIIB)""",null,null,"""YES""","""Genetically engineered form of…",null,null,null,null,"""AIDS drug""",null
"""sT4""",null,null,"""YES""",null,"""Recombinant Soluble T4""",null,null,null,"""AIDS drug""",null


In [ ]:
regimens

Name,EVS ID,Histology,Radiation,Remarks,Primary Site,Alternate Names,Drugs
str,str,str,str,str,str,str,str
"""7+3""",null,null,null,"""6/12/23 Moved to hidden status""",null,null,"""Mitoxantrone;Daunorubicin Hydr…"
"""ABCD""",null,null,null,null,null,"""ADBC""","""Dacarbazine;Lomustine;Doxorubi…"
"""ABCM""",null,null,null,null,null,null,"""Mitomycin;Doxorubicin;Cyclopho…"
"""ABD""",null,null,null,null,null,null,"""Dacarbazine;Doxorubicin;Bleomy…"
"""ABDIC""",null,null,null,null,null,null,"""Dacarbazine;Lomustine;Doxorubi…"
…,…,…,…,…,…,…,…
"""ara-C + DNR + PRED + MP""",null,null,null,null,null,null,"""Daunorubicin Hydrochloride;Mer…"
"""ara-C-HU""",null,null,null,null,null,null,"""Hydroxyurea"""
"""ara-C/MTX""",null,null,null,null,null,null,"""Methotrexate;Cytarabine;Filgra…"


# regex to find series with know drugs

for starters print from subset a row for each series id in manual. ie, subset -> sample one per expeirment -> print the series id, title, overall design

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's how you can do that:

```python
score_subset.filter(pl.col('experiment_accession').is_in(manual_ids)).unique('experiment_accession').select(['experiment_accession', 'series_title', 'series_overall_design'])
```

This will:
1. Filter to only the experiments in your `manual_ids` list
2. Keep one unique row per experiment
3. Select just the three columns you want

Would you like to try running this?

In [ ]:
score_subset.filter(pl.col('experiment_accession').is_in(manual_ids)).unique('experiment_accession').select(['experiment_accession', 'series_title', 'series_overall_design'])

experiment_accession,series_title,series_overall_design
str,str,str
"""GSE29561""","""Response prediction to neoadju…","""22 Samples from breast cancer …"
"""GSE28583""","""Differentially expressed genes…","""After informed consent, patien…"
"""GSE31852""","""An EGFR-mutation signature rev…","""Gene expression profiles were …"
"""GSE95770""","""Sorafenib promotes graft-versu…","""Peripheral blood mononuclear c…"
"""GSE93984""","""Expression data from DLBCL tum…","""Gene expression in ibrutinib p…"
…,…,…
"""GSE15622""","""Expression data from the CTCR-…","""Patients with suspected ovaria…"
"""GSE28694""","""Differentially expressed genes…","""After informed consent, patien…"
"""GSE106977""","""Triple negative breast cancer …","""We performed a retrospective a…"


In [ ]:
for row in score_subset.filter(pl.col('experiment_accession').is_in(manual_ids)).unique('experiment_accession').iter_rows(named=True):
    print(f"\n{row['experiment_accession']}")
    print(f"Title: {row['series_title']}")
    print(f"Design: {row['series_overall_design']}")


GSE29561
Title: Response prediction to neoadjuvant chemotherapy
Design: 22 Samples from breast cancer patients with corresponding treatment response indicators

GSE18864
Title: Tumor expression data from neoadjuvant trial of cisplatin monotherapy in triple negative breast cancer patients
Design: Pretreatment tumor samples from the clinical trial (N=24 with adequate tissue) were used for RNA extraction, linear amplification, biotin labeling and hybridization to Affymetrix U133 plus 2.0 arrays.  A reference set of 51 primary breast tumors representing all subtypes of breast cancer were processed in a similar manner to include linear amplification, and hybridized to Affymetrix arrays.

GSE28826
Title: Differentially expressed genes after treatment with chemotherapy in breast cancer and their correlation with pathologic bad response (Miller & Payne grades 1 and 2)
Design: After informed consent, patients with a histologically confirmed diagnosis of breast cancer and scheduled chemotherapy

now , i want to build a pipeline that extract the drugs in these collumns. 

a few examples:
GSE93984 -> 'ibrutinib'

GSE106977 -> 'anthracyclines', 'taxens', 'carboplatin'

some may require looking at the sample level data: but that can wait
GSE29561 -> 'neo adjuvant'

GSE15622 -> 'Carboplatin', 'Paclitaxel'


consider how the reginamens and drugs are listed in the tables of drugs and regimens. what challenges are there in running this kind of regex? 

for now, lets focus on the drugs, later we will figutre out how to convert regimens to drugs


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question! Let's think through the challenges before writing code.

Looking at the `drugs` dataframe, I can see a few potential issues:

**Challenges with regex matching:**

1. **Case sensitivity** - "Carboplatin" vs "carboplatin" in text
2. **Word boundaries** - Don't want "platinum" to match when searching for specific platinum drugs
3. **Special characters** - Drug names like `5-Aza-2'Deoxy-cytidine` have hyphens, numbers, apostrophes
4. **Short names/abbreviations** - The `Abbreviation` column has things like "DAC" which could cause false positives
5. **Alternate names** - There's an `Alternate Names` column you might want to search too
6. **Drug classes vs individual drugs** - "anthracyclines" and "taxanes" are *classes*, not in the drugs list (those would be Doxorubicin, Epirubicin, Paclitaxel, Docetaxel, etc.)

Take a look at what columns/name variants are available:

```python
drugs.select(['Name', 'Abbreviation', 'Alternate Names']).head(20)
```

What do you think would be a good starting approach? A few options:

- Start simple with just the `Name` column, case-insensitive
- Build a combined list of all name variants (Name + Abbreviation + Alternate Names)
- Filter to only "longer" drug names first to avoid false positives from short abbreviations

Which direction would you like to explore first?

In [ ]:
drugs.select(['Name', 'Abbreviation', 'Alternate Name','Category','Sub-category']).head(20)

Name,Abbreviation,Alternate Name,Category,Sub-category
str,str,str,str,str
"""131I-MIBG""",null,"""I131 metaiodobenzylguanidine""","""Radiation""","""Radioisotope"""
"""5-Aza-2'Deoxy-cytidine""","""DAC""","""Decytabine""","""Chemotherapy""","""Cytostatic agent--DNA methyl t…"
"""5-FP""",null,"""5-fluoro-Pyrimidinone""","""Chemotherapy""","""Antimetabolite"""
"""ABT 888""",null,"""Veliparib""","""Chemotherapy""","""PARP Inhibitor"""
"""ADXS-HPV""",null,null,"""Biologic therapy (BRM, immunot…",null
…,…,…,…,…
"""Acetylcysteine""",null,null,"""Ancillary Agent""","""Ancillary"""
"""Acitretin""",null,"""Etretin;Ro 10-1670;Soriatane""","""Differentiation inducing agent""",null
"""Acyclovir""",null,"""Acycloguanosine;Avirax;BW-248U…","""Ancillary Agent""",null


i think we should start with a case insentitive regex, matching full name. during search i think we sould collapse hyphens. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking! Collapsing hyphens will help with inconsistent formatting (e.g., "5-FU" vs "5 FU" vs "5FU").

A few quick clarifications on "collapse hyphens":

1. **Replace hyphens with spaces** in both drug name and search text, then match?
2. **Remove hyphens entirely** from both?
3. **Make hyphens optional** in the regex pattern (e.g., `5-?FU` matches "5-FU" or "5FU")?

Option 2 (remove entirely) is simplest and would handle most cases. You'd normalize both the drug names and the text before matching.

Want to start by:
1. Building a list of normalized drug names (lowercase, hyphens removed)
2. Then writing a function that normalizes text the same way and searches for matches?

lets remove hyphens from both durign search

In [ ]:
drugs = drugs.with_columns(
    pl.col('Name').str.to_lowercase().str.replace_all('-', '').alias('name_normalized')
)

we will be doing regex matching many patterns to many rows. what is the most efficient way of doing this?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question! A few approaches, from simpler to more optimized:

1. **Naive loop** - check each pattern against each row (slow: O(patterns × rows))

2. **Combined regex with alternation** - join all patterns into one: `(drug1|drug2|drug3|...)` and search once per row. Single pass through text per row.

3. **Aho-Corasick algorithm** - specifically designed for multi-pattern string matching. Builds a trie/automaton from all patterns, then searches text in a single pass. Very fast for many patterns. Python has `pyahocorasick` library.

For ~1800 drugs and thousands of rows, option 2 (combined regex) is probably the sweet spot of simplicity and performance. Polars' `str.contains` or `str.extract_all` with a single combined pattern would be efficient.

Which approach would you like to try?

approach 2 it is. 
so i think we need to :
1. build the regexp patterns
2. buld a search function that can be applied per row. the function also collapses hyphens so that they don't infulence the search. 

what else should i htink about?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good plan! A few more things to consider:

**Word boundaries** - Should "platinum" match inside "carboplatin"? Probably not. You'll want `\b` word boundaries around patterns: `\b(drug1|drug2|...)\b`

**Regex escaping** - Some drug names may contain special regex characters like parentheses, e.g., `Ad5CMV-p53 (adenovirus p53)`. Need to escape those with `re.escape()`.

**Short names** - Some abbreviations might be very short (2-3 chars). Could cause false positives. Maybe set a minimum length threshold, or handle abbreviations separately?

**Return value** - Do you want:
- Boolean (any match found)?
- List of matched drug names?
- Both?

**Which columns to search** - Starting with `series_title` + `series_overall_design`? Concatenate them?

Anything else you want to add, or ready to start building the pattern?

In [ ]:

for drug in drugs['Name']:
    print(drug)

131I-MIBG
5-Aza-2'Deoxy-cytidine
5-FP
ABT 888
ADXS-HPV
AMG 531
ATRA-IV
Abecma
Abemaciclib
Abiraterone acetate
Abraxane
Acalabrutinib
Accutane
Acetaminophen
Acetazolamide
Acetylcysteine
Acitretin
Acyclovir
Acyclovir Sodium
Ad5CMV-p53 (adenovirus p53)
Ado-trastuzumab emtansine
Adrenocorticotropic hormone
Adstiladrin
Afinitor
Akeega
Albendazole
Albuterol Sulfate
Alectinib
Alemtuzumab
Alendronate Sodium
Alitretinoin
Allopurinol
Allosteric ErbB Inhibitor BDTX-189
Alpharadin
Altretamine
Amifostine
Aminocamptothecin
Aminoglutethimide
Aminopterin
Aminosyn
Amtagvil
Anagrelide HCl
Anastrozole
Ancitabine Hydrochloride
Anktiva
Antithymocyte Globulin
Apaziquone
Aprepitant
Asparaginase
Asparaginase (Erwinia carotovora)
Asparaginase-PEG
Asparaginase-PEG (K-H)
Asparlas
Aspirin
Atevirdine Mesylate
Atezolizumab
Atovaquone
Atripla
Aucatzyl
Aut Tum Vac/BCG/IL-2
Aut Tum Vac/IL-2/BCG
Aut Tum Vac/IL-2/BCG, Tice BCG Vaccine, frozen
Autologous Tumor Cell (BCG + IL-2 (Cetus),
Autologous Tumor Cell (BCG) + IL-2 

for now match with word boundaries. 
in the regex construction -> only single words (after hyphen collpase). lets ignore parenthasies. 
for now don't use abbreviations
return list of matched and a boolean

initially - search all collumn of orginial ds

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect! So to clarify the plan:

**Pattern construction:**
- Filter drugs to only those with single words (no spaces after hyphen removal)
- Use `re.escape()` on each drug name
- Join with `|` and wrap with `\b(...)\b`

**Search function:**
- Takes a row (or text)
- Normalizes: lowercase + remove hyphens
- Searches with the pattern
- Returns: `(boolean, list_of_matches)`

**Apply to:**
- All text columns in the original dataset (which dataset - `samples` or `score_subset`?)

Should I write the pattern construction code, or would you like to try building it first?

help me build the regex. lets use the rexep library

In [ ]:
import regex as re

# Filter to single-word drug names
single_word_drugs = drugs.filter(
    pl.col('name_normalized').str.contains(' ').not_()
)
len(single_word_drugs)

1230

okay, that seems fine. now we need to create the pattern. lets use the fastcore L with mapping method. 
1. pass to L
2. use .map(re.escape)
3. join with '|' 
4. wrap with \b

In [ ]:
pattern_parts = L(list(single_word_drugs['name_normalized'])).map(re.escape)
pattern = r'\b(' + '|'.join(pattern_parts) + r')\b'

In [ ]:
pattern[:100]

"\\b(131imibg|5aza2'deoxycytidine|5fp|adxshpv|atraiv|abecma|abemaciclib|abraxane|acalabrutinib|accutan"

In [ ]:
def search_all_columns_with_cols(row):
    all_matches = []
    match_cols = []
    for col, val in row.items():
        if val is not None:
            found, drugs_list = find_drugs(str(val))
            if found:
                all_matches.extend(drugs_list)
                match_cols.append(col)
    return (len(all_matches) > 0, list(set(all_matches)), list(set(match_cols)))

In [ ]:

results = []
for row in samples.iter_rows(named=True):
    found, drugs_list, cols_list = search_all_columns_with_cols(row)
    if found:
        results.append({
            'experiment_accession': row['experiment_accession'],
            'refinebio_accession_code': row['refinebio_accession_code'],
            'matched_drugs': drugs_list,
            'matched_columns': cols_list
        })

results_df = pl.DataFrame(results)

In [ ]:
results_df = results_df.sort('experiment_accession')

In [ ]:
results_df 

experiment_accession,refinebio_accession_code,matched_drugs,matched_columns
str,str,list[str],list[str]
"""GSE106977""","""GSM2858725""","[""carboplatin""]","[""series_overall_design""]"
"""GSE106977""","""GSM2858712""","[""carboplatin""]","[""series_overall_design""]"
"""GSE106977""","""GSM2858782""","[""carboplatin""]","[""series_overall_design""]"
"""GSE106977""","""GSM2858703""","[""carboplatin""]","[""series_overall_design""]"
"""GSE106977""","""GSM2858717""","[""carboplatin""]","[""series_overall_design""]"
…,…,…,…
"""GSE95770""","""GSM2525644""","[""sorafenib""]","[""description"", ""series_title"", … ""series_overall_design""]"
"""GSE95770""","""GSM2525653""","[""sorafenib""]","[""description"", ""series_title"", … ""series_overall_design""]"
"""GSE95770""","""GSM2525642""","[""sorafenib""]","[""description"", ""series_title"", … ""series_overall_design""]"


now create a subset of samples, keeping only these gsm ids.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You can filter `samples` to keep only the matching GSM IDs like this:

```python
matched_samples = samples.filter(
    pl.col('refinebio_accession_code').is_in(results_df['refinebio_accession_code'])
)
```

Would you like to run this and check the shape of the filtered dataset?

In [ ]:
drugged_samples = samples.filter(
    pl.col('refinebio_accession_code').is_in(results_df['refinebio_accession_code'])
)

/tmp/ipykernel_276/3334715526.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  drugged_samples = samples.filter(


# identify response collumns

In [ ]:

# 1) Canonical response codes/labels (values)
RESPONSE_VALUE_TOKENS: Set[str] = {
    # atomic codes
    "PCR","RD","CR","NR","R","SD","PPR","PNC","PD","CRU","IR","PRCR","NON-CR","VGPR","NCR","PR","CRP",
    "MLFS","HI","RCB","CRI","OJBR","NOR","NC","ICCR","ICPR","ISD","ICPD","IUPD","IUPR","NONE","S","HR","TF","ED",
    # common long forms / keywords
    "PATHOLOGICAL COMPLETE REMISSION","RESIDUAL DISEASE","COMPLETE RESPONSE","NONRESPONSE","RESPONSE","STABLE DISEASE",
    "PATHOLOGICAL PARTIAL RESPONDERS","NON-RESPONDERS","PROGRESSIVE DISEASE","UNCONFIRMED COMPLETED REMISSION",
    "INCOMPLETE RESPONSE","LONG-HER","PARTIAL RESPONSE AND COMPLETE RESPONSE","NON-COMPLETE REMISSION",
    "VERY GOOD PARTIAL RESPONSE","GOOD/COMPLETE PCR","CR WITH INCOMPLETE HEMATOLOGIC OR PLATELET RECOVERY",
    "MORPHOLOGIC LEUKEMIA-FREE STATE","HEMATOLOGIC IMPROVEMENT","RESIDUAL CANCER BURDEN",
    "COMPLETE REMISSION WITH INCOMPLETE HEMATOLOGIC RECOVERY","OBJECTIVE RESPONDERS","NON-RESPONDERS","NO CHANGE",
    "CONFIRMED COMPLETE RESPONSE","CONFIRMED PARTIAL RESPONSE","CONFIRMED PROGRESSIVE DISEASE",
    "UNCONFIRMED PROGRESSIVE DISEASE","UNCONFIRMED PARTIAL RESPONSE","NO RESPONSE","SENSITIVE",
    "HIGHLY RESISTANT","TREATMENT FAILURE","EARLY DEATH","RESIDUAL","SURVIVAL","PROGRESSIVE",
    # generic binaries
    "YES","NO","Y","N","TRUE","FALSE","T","F","1","0"
}

RESPONSE_VALUE_TOKENS: Set[str] = {
    # atomic codes
    "PCR","RD","CR","NR","R","SD","PPR","PNC","PD","CRU","IR","PRCR","NON-CR","VGPR","NCR","PR","CRP",
    "MLFS","HI","RCB","CRI","OJBR","NOR","NC","ICCR","ICPR","ISD","ICPD","IUPD","IUPR","NONE","S","HR","TF","ED",
    # common long forms / keywords
    "PATHOLOGICAL COMPLETE REMISSION","RESIDUAL DISEASE","COMPLETE RESPONSE","NONRESPONSE","RESPONSE","STABLE DISEASE",
    "PATHOLOGICAL PARTIAL RESPONDERS","NON-RESPONDERS","PROGRESSIVE DISEASE","UNCONFIRMED COMPLETED REMISSION",
    "INCOMPLETE RESPONSE","PARTIAL RESPONSE AND COMPLETE RESPONSE","NON-COMPLETE REMISSION",
    "VERY GOOD PARTIAL RESPONSE","GOOD/COMPLETE PCR","CR WITH INCOMPLETE HEMATOLOGIC OR PLATELET RECOVERY",
    "MORPHOLOGIC LEUKEMIA-FREE STATE",
    "COMPLETE REMISSION WITH INCOMPLETE HEMATOLOGIC RECOVERY","OBJECTIVE RESPONDERS","NON-RESPONDERS",
    "CONFIRMED COMPLETE RESPONSE","CONFIRMED PARTIAL RESPONSE","CONFIRMED PROGRESSIVE DISEASE",
    "UNCONFIRMED PROGRESSIVE DISEASE","UNCONFIRMED PARTIAL RESPONSE","NO RESPONSE","SENSITIVE",
    "HIGHLY RESISTANT","TREATMENT FAILURE","RESIDUAL","PROGRESSIVE",
    # generic binaries
    "YES","NO","Y","N","TRUE","FALSE","T","F","1","0"
}

# Atomic codes for regex matching (short codes that need word boundary matching)
ATOMIC_CODES: Set[str] = {
    "PCR","RD","CR","NR","R","SD","PPR","PNC","PD","CRU","IR","PRCR","NON-CR","VGPR","NCR","PR","CRP",
    "MLFS","HI","RCB","CRI","OJBR","NOR","NC","ICCR","ICPR","ISD","ICPD","IUPD","IUPR","NONE",
}

# Long forms require exact match (already complete phrases)
LONG_FORMS: Set[str] = RESPONSE_VALUE_TOKENS - ATOMIC_CODES

# Compile regex pattern for atomic codes with word boundaries
# Pattern: (^|[^A-Z0-9])CODE([^A-Z0-9]|$)
# This matches 'PCR' in 'PCR 1', 'PCR:yes', 'pcr-positive' but not 'PCRTEST'
_ATOMIC_PATTERN = (
    r'(?:^|[^A-Z0-9])(' +
    '|'.join(re.escape(code) for code in ATOMIC_CODES) +
    r')(?:[^A-Z0-9]|$)'
)
ATOMIC_REGEX = re.compile(_ATOMIC_PATTERN, re.IGNORECASE)

# 2) Column-name terms (be specific; avoid single-letter codes here)
NAME_TERMS: Set[str] = {
    # core signals
    "RESPONSE","RESPONDER","RESPONDERS","RESPONDING","RESP","RESPONSES",
    "REMISSION","COMPLETE_REMISSION","NON_COMPLETE_REMISSION","NON-COMPLETE_REMISSION",
    "PROGRESSIVE_DISEASE","STABLE_DISEASE","PARTIAL_RESPONSE","COMPLETE_RESPONSE",
    "PROGRESSION","BURDEN","RESIDUAL","PATHOLOGICAL","PATH","RCB",
    # frequent clinical abbreviations safe for NAMES
    "PCR","CR","PR","SD","PD","VGPR","CRU","CRI","CRP","MLFS","HI","OJBR",
    "ICCR","ICPR","ISD","ICPD","IUPD","IUPR","NR","NONRESPONDER","NONRESPONDERS","NON_RESPONSE",
    "SENSITIVE","RESISTANT","RESISTANCE","TREATMENT_FAILURE","FAILURE","EARLY_DEATH","ED",
    # words that often wrap the above
    "OUTCOME","EFFICACY","EFFECTIVENESS",
    "RESP_CAT","RESP_CLASS","RESP_STATUS","CLINICAL_BENEFIT","OBJECTIVE_RESPONSE",
    "RECIST","BOR"  # Best Overall Response
    #words that are too generic for NAMES (excluded): "THERAPEUTIC","THERAPY","TREATMENT",
    
}

def _compile_name_regex(terms: Iterable[str]) -> re.Pattern:
    """
    Build a robust column-name regex from NAME_TERMS.
    - Normalizes underscores/hyphens vs words
    - Word-boundaries to avoid overmatching substrings
    """
    esc = []
    for t in terms:
        t = t.strip().upper().replace("-", "[\\-_]").replace("_", "[\\-_]")
        esc.append(t)
    # word-ish boundaries: (^|[^A-Z0-9]) ... (?=[^A-Z0-9]|$)
    pattern = r"(?i)(^|[^A-Z0-9])(" + "|".join(esc) + r")(?=[^A-Z0-9]|$)"
    return re.compile(pattern)

NAME_RE = _compile_name_regex(NAME_TERMS)

def normalize_token(x: str) -> str:
    """Uppercase + collapse spaces + keep A-Z0-9/+ and '-' (remove punctuation)."""
    x = (x or "").strip().upper()
    x = re.sub(r"[^A-Z0-9/\-\s]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

In [ ]:
TEXT_SEARCH_COLUMNS = [
    # Series metadata - Experiment-level context (can be EITHER cell line OR primary human)
    'series_title',
    'series_summary',
    'series_overall_design',
    
    # refinebio columns - EXPANDED with more refinebio columns
    'refinebio_title',
    'refinebio_description', 
    'refinebio_source_name',
    'refinebio_subject',
    'refinebio_age',
    'refinebio_cell_line',
    'refinebio_disease',
    'refinebio_disease_stage',
    'refinebio_sex',
    'refinebio_source_database',
    'refinebio_specimen_part',
    'refinebio_treatment',
    # GEO protocol columns (strong cell line indicators)
    'description',
    'extraction_protocol_ch1',
    'growth_protocol_ch1',
    'treatment_protocol_ch1',
]

now i want to extend the drugged sampled df by applying a similar logic for matching drugs, but this time, matching the contents to regexp inidicating a repsonse value. 
also, pull the collumn names that mantch the NAME_RE, and and the df should hold a collumn of whihc of those wasn't null for that smaple

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let me make sure I understand what you want:


1. **Match column names** - find which columns in the dataset have names matching `NAME_RE` (response-related column names)
2. **Track non-null response columns per sample** - for each row, list which of those response-named columns actually have a value
3. **Match response values in content** - similar to the drug matching, search chosen column values for response codes/keywords (using `ATOMIC_REGEX` and `LONG_FORMS`):
    chosen collumns: the column named matched, and the TEXT_SEARCH_COLUMNS


So the final df would have new columns like:
- `matched_response_values` - list of response codes found in any column content
- `matched_respone_columns` - a list of the columns in which the matches were found
- `response_columns` - list of column names that matched `NAME_RE` and were non-null for that sample

Is that right? 

for now, we will apply this this the drugged_samples only and extend the results df obtained in the previous part

yes, that good

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great! Let's build this step by step. First, let's create a function to find response values in text:

```python
def find_response_values(text):
    if text is None: return (False, [])
    text_norm = normalize_token(str(text))
    matches = set()
    for match in ATOMIC_REGEX.finditer(text_norm): matches.add(match.group(1))
    for long_form in LONG_FORMS:
        if long_form in text_norm: matches.add(long_form)
    return (len(matches) > 0, list(matches))
```

Now let's find which columns have response-related names:

```python
response_col_names = [col for col in drugged_samples.columns if NAME_RE.search(col)]
response_col_names
```

In [ ]:
def find_response_values(text):
    if text is None: return (False, [])
    text_norm = normalize_token(str(text))
    matches = set()
    for match in ATOMIC_REGEX.finditer(text_norm): matches.add(match.group(1))
    for long_form in LONG_FORMS:
        if long_form in text_norm: matches.add(long_form)
    return (len(matches) > 0, list(matches))

In [ ]:
response_col_names = [col for col in drugged_samples.columns if NAME_RE.search(col)]
response_col_names

['characteristics_ch1_PR',
 'characteristics_ch1_Pathological M',
 'characteristics_ch1_Pathological N',
 'characteristics_ch1_Pathological T',
 'characteristics_ch1_Response',
 'characteristics_ch1_Response.to.treatment ',
 'characteristics_ch1_ac response',
 'characteristics_ch1_best response (if different)',
 'characteristics_ch1_birth outcome (sptb 1, term 0)',
 'characteristics_ch1_birthing outcome (sptl 1, pprom 2, term 3)',
 'characteristics_ch1_bv6 response',
 'characteristics_ch1_category (0 = normal, 1 = hpv without progression, 2 = hpv with progression)',
 'characteristics_ch1_chemotherapy response',
 'characteristics_ch1_cisplatin response (complete response or incomplete response)',
 'characteristics_ch1_clinical outcome',
 'characteristics_ch1_clinical status post 1st line chemotherapy (cr - complete response; pr - partial response; sd - stable disease; p - progression)',
 'characteristics_ch1_code progression-free survival',
 'characteristics_ch1_date of progression',
 '

carry on

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Now let's build the function to search for response values across columns and extend the results:

```python
def search_response_info(row):
    resp_cols_with_values = [col for col in response_col_names if row.get(col) is not None]
    search_cols = list(set(resp_cols_with_values + TEXT_SEARCH_COLUMNS))
    all_matches, match_cols = [], []
    for col in search_cols:
        val = row.get(col)
        if val is not None:
            found, resp_list = find_response_values(str(val))
            if found:
                all_matches.extend(resp_list)
                match_cols.append(col)
    return (list(set(all_matches)), list(set(match_cols)), resp_cols_with_values)
```

Now apply it to extend the results:

```python
extended_results = []
for row in drugged_samples.iter_rows(named=True):
    gsm = row['refinebio_accession_code']
    if gsm in results_df['refinebio_accession_code']:
        resp_vals, resp_match_cols, resp_cols = search_response_info(row)
        drug_info = results_df.filter(pl.col('refinebio_accession_code') == gsm).row(0, named=True)
        extended_results.append(dict(experiment_accession=drug_info['experiment_accession'], refinebio_accession_code=gsm, matched_drugs=drug_info['matched_drugs'], drug_matched_columns=drug_info['matched_columns'], matched_response_values=resp_vals, matched_response_columns=resp_match_cols, response_columns=resp_cols))

extended_results_df = pl.DataFrame(extended_results)
extended_results_df
```

In [ ]:
def search_response_info(row):
    resp_cols_with_values = [col for col in response_col_names if row.get(col) is not None]
    search_cols = list(set(resp_cols_with_values + TEXT_SEARCH_COLUMNS))
    all_matches, match_cols = [], []
    for col in search_cols:
        val = row.get(col)
        if val is not None:
            found, resp_list = find_response_values(str(val))
            if found:
                all_matches.extend(resp_list)
                match_cols.append(col)
    return (list(set(all_matches)), list(set(match_cols)), resp_cols_with_values)

In [ ]:
extended_results = []
for row in drugged_samples.iter_rows(named=True):
    gsm = row['refinebio_accession_code']
    if gsm in results_df['refinebio_accession_code']:
        resp_vals, resp_match_cols, resp_cols = search_response_info(row)
        drug_info = results_df.filter(pl.col('refinebio_accession_code') == gsm).row(0, named=True)
        extended_results.append(dict(experiment_accession=drug_info['experiment_accession'], refinebio_accession_code=gsm, matched_drugs=drug_info['matched_drugs'], drug_matched_columns=drug_info['matched_columns'], matched_response_values=resp_vals, matched_response_columns=resp_match_cols, response_columns=resp_cols))

extended_results_df = pl.DataFrame(extended_results)
extended_results_df

experiment_accession,refinebio_accession_code,matched_drugs,drug_matched_columns,matched_response_values,matched_response_columns,response_columns
str,str,list[str],list[str],list[str],list[str],list[str]
"""GSE41998""","""GSM1030127""","[""paclitaxel"", ""doxorubicin"", ""cyclophosphamide""]","[""series_title"", ""series_summary"", ""series_overall_design""]","[""PCR"", ""N"", … ""0""]","[""matched_response_values"", ""refinebio_sex"", … ""response_columns""]","[""characteristics_ch1_ac response"", ""characteristics_ch1_pcr"", … ""response_columns""]"
"""GSE42296""","""GSM1037347""","[""infliximab""]","[""treatment_protocol_ch1"", ""series_summary"", ""series_title""]","[""N"", ""TRUE"", … ""0""]","[""matched_response_values"", ""refinebio_subject"", … ""characteristics_ch1_response""]","[""characteristics_ch1_response"", ""matched_response_values"", … ""response_columns""]"
"""GSE50614""","""GSM1224700""","[""etanercept""]","[""series_summary""]","[""N"", ""RESIDUAL"", … ""0""]","[""matched_response_values"", ""refinebio_subject"", … ""response_columns""]","[""characteristics_ch1_histological response"", ""matched_response_values"", … ""response_columns""]"
"""GSE65095""","""GSM1587074""","[""docetaxel"", ""vinorelbine"", … ""epirubicin""]","[""series_summary""]","[""N"", ""TRUE"", … ""0""]","[""matched_response_values"", ""series_title"", … ""characteristics_ch1_pr_status""]","[""characteristics_ch1_pr_status"", ""matched_response_values"", … ""response_columns""]"
"""GSE73578""","""GSM1898147""","[""prednisone""]","[""series_overall_design""]","[""N"", ""TRUE"", … ""0""]","[""matched_response_values"", ""refinebio_sex"", … ""response_columns""]","[""characteristics_ch1_prednisone poor responder"", ""matched_response_values"", … ""response_columns""]"
…,…,…,…,…,…,…
"""GSE65095""","""GSM1587155""","[""docetaxel"", ""vinorelbine"", … ""epirubicin""]","[""series_summary""]","[""N"", ""TRUE"", … ""0""]","[""matched_response_values"", ""series_title"", … ""characteristics_ch1_pr_status""]","[""characteristics_ch1_pr_status"", ""matched_response_values"", … ""response_columns""]"
"""GSE73578""","""GSM1898200""","[""prednisone""]","[""series_overall_design""]","[""N"", ""TRUE"", … ""0""]","[""matched_response_values"", ""series_title"", … ""response_columns""]","[""characteristics_ch1_prednisone poor responder"", ""matched_response_values"", … ""response_columns""]"
"""GSE93984""","""GSM2466134""","[""ibrutinib""]","[""series_overall_design"", ""series_summary""]","[""STABLE DISEASE"", ""N"", … ""S""]","[""matched_response_values"", ""characteristics_ch1_progression_free_survival_censor"", … ""characteristics_ch1_response""]","[""characteristics_ch1_progression_free_survival"", ""characteristics_ch1_progression_free_survival_censor"", … ""response_columns""]"


In [ ]:
series_grouped = extended_results_df.drop('refinebio_accession_code').group_by('experiment_accession').agg([
    pl.col('matched_drugs').flatten().unique(),
    pl.col('drug_matched_columns').flatten().unique(),
    pl.col('matched_response_values').flatten().unique(),
    pl.col('matched_response_columns').flatten().unique(),
    pl.col('response_columns').flatten().unique(),
])

In [ ]:
series_grouped 

experiment_accession,matched_drugs,drug_matched_columns,matched_response_values,matched_response_columns,response_columns
str,list[str],list[str],list[str],list[str],list[str]
"""GSE65095""","[""docetaxel"", ""vinorelbine"", … ""cyclophosphamide""]","[""series_summary""]","[""HR"", ""T"", … ""TF""]","[""refinebio_disease_stage"", ""characteristics_ch1_pr_status"", … ""series_summary""]","[""matched_response_values"", ""matched_response_columns"", … ""characteristics_ch1_pr_status""]"
"""GSE57383""","[""infliximab""]","[""series_title""]","[""RESPONSE"", ""N"", … ""ED""]","[""refinebio_age"", ""characteristics_ch1_response"", … ""refinebio_title""]","[""characteristics_ch1_response"", ""response_columns"", … ""matched_response_values""]"
"""GSE33377""","[""infliximab""]","[""refinebio_treatment"", ""characteristics_ch1_treatment""]","[""TF"", ""ED"", … ""NO""]","[""characteristics_ch1_response"", ""matched_response_values"", … ""series_summary""]","[""matched_response_columns"", ""characteristics_ch1_response"", … ""matched_response_values""]"
"""GSE58598""","[""tamoxifen""]","[""refinebio_treatment"", ""characteristics_ch1_treatment"", ""series_summary""]","[""TRUE"", ""T"", … ""F""]","[""description"", ""refinebio_sex"", … ""refinebio_treatment""]","[""matched_response_values"", ""characteristics_ch1_response"", … ""matched_response_columns""]"
"""GSE80077""","[""letrozole"", ""progesterone"", ""tamoxifen""]","[""treatment_protocol_ch1"", ""series_summary""]","[""F"", ""0"", … ""NO""]","[""treatment_protocol_ch1"", ""characteristics_ch1_ihc.pr"", … ""series_title""]","[""response_columns"", ""characteristics_ch1_pathological ihc er"", … ""characteristics_ch1_pathological ihc her2""]"
…,…,…,…,…,…
"""GSE85047""","[""st4""]","[""characteristics_ch1_inss""]","[""N"", ""HR"", … ""NO""]","[""matched_response_values"", ""refinebio_title"", … ""series_summary""]","[""matched_response_columns"", ""characteristics_ch1_event_progression_free"", … ""response_columns""]"
"""GSE41998""","[""doxorubicin"", ""cyclophosphamide"", ""paclitaxel""]","[""characteristics_ch1_treatment arm"", ""series_overall_design"", … ""series_summary""]","[""ED"", ""RESPONSE"", … ""PROGRESSIVE DISEASE""]","[""refinebio_sex"", ""characteristics_ch1_pcr"", … ""response_columns""]","[""matched_response_columns"", ""characteristics_ch1_pcr"", … ""matched_response_values""]"
"""GSE18864""","[""cisplatin""]","[""series_summary"", ""source_name_ch1"", ""series_title""]","[""N"", ""F"", … ""1""]","[""series_overall_design"", ""matched_response_columns"", … ""matched_response_values""]","[""matched_response_values"", ""characteristics_ch1_miller-payne response"", … ""matched_response_columns""]"


In [ ]:
def inspect_series(exp_id, n=5):
    series_info = series_grouped.filter(pl.col('experiment_accession') == exp_id).row(0, named=True)
    series_meta = score_subset.filter(pl.col('experiment_accession') == exp_id).unique('experiment_accession').row(0, named=True)
    print(f"=== {exp_id} ===\n")
    print(f"Title: {series_meta['series_title']}")
    print(f"Design: {series_meta['series_overall_design'][:200]}...\n")
    print(f"Matched drugs: {series_info['matched_drugs']}")
    print(f"Drug columns: {series_info['drug_matched_columns']}")
    print(f"Response values: {series_info['matched_response_values']}")
    resp_cols = [c for c in series_info['response_columns'] if c not in ['matched_response_values', 'response_columns', 'matched_response_columns']]
    print(f"Response columns: {resp_cols}\n")
    all_cols = list(set(series_info['drug_matched_columns'] + resp_cols))
    samples = extended_results_df.filter(pl.col('experiment_accession') == exp_id).head(n)
    for row in samples.iter_rows(named=True):
        print(f"\n--- {row['refinebio_accession_code']} ---")
        sample_row = drugged_samples.filter(pl.col('refinebio_accession_code') == row['refinebio_accession_code']).row(0, named=True)
        for col in all_cols:
            val = sample_row.get(col)
            if val is not None: print(f"  {col}: {val}")

In [ ]:
inspect_series('GSE31852', n=1)


=== GSE31852 ===

Title: An EGFR-mutation signature reveals features of the EGFR-dependent phenotype and identifies MACC1 as an EGFR-associated regulator of MET.
Design: Gene expression profiles were measured in 124 core biopsies from patients with refractory non-small cell lung cancer in the  Biomarker-integrated Approaches of Targeted Therapy for Lung Cancer Elimina...

Matched drugs: ['erlotinib', 'pemetrexed']
Drug columns: ['refinebio_treatment', 'series_summary', 'characteristics_ch1_treatment']
Response values: ['F', 'NO', 'S', 'N', '1', 'T', '0', 'ED', 'Y', 'TRUE', 'TF', 'RESPONSE']
Response columns: ['characteristics_ch1_date of progression', 'characteristics_ch1_progression-free survival status', 'characteristics_ch1_progression-free survival time (months)']


--- GSM790028 ---
  characteristics_ch1_progression-free survival time (months): 5.5852
  characteristics_ch1_date of progression: 2009-10-22
  refinebio_treatment: erlotinib
  characteristics_ch1_treatment: erlotinib
 

now create a text file:

it runs the above on each series in the group, and creates a final report (.md file) of series id, and two samples. i will pass this for evaluation